In [1]:
# Setup
import os, sys, json, time, re, math
from pathlib import Path
from getpass import getpass
from datetime import datetime
from collections import defaultdict

# Detect environment
IS_COLAB = False
IS_KAGGLE = False
ENV_NAME = "Local"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    ENV_NAME = "Colab"
    BASE_PATH = Path('/content/drive/MyDrive/AGoT-ReAct/Math Performance')
except:
    # Check if Kaggle
    if os.path.exists('/kaggle/working'):
        IS_KAGGLE = True
        ENV_NAME = "Kaggle"
        BASE_PATH = Path('/kaggle/working')
    else:
        BASE_PATH = Path(r'f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance')

print(f"{ENV_NAME} Environment | {BASE_PATH}")

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("⚠️ No GPU detected - model will run on CPU (slower)")
except:
    print("⚠️ PyTorch not installed - installing dependencies...")

# Install minimal deps
if IS_COLAB or IS_KAGGLE:
    os.system('pip install -q transformers torch accelerate datasets tqdm beautifulsoup4 requests bitsandbytes')
else:
    print("Installing dependencies locally...")
    os.system('pip install -q transformers torch accelerate datasets tqdm beautifulsoup4 requests')

Kaggle Environment | /kaggle/working
✓ GPU available: Tesla T4
  Memory: 15.8 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
from tqdm import tqdm
import requests
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from dotenv import load_dotenv
    load_dotenv()
except:
    pass

# Paths & config
OUTPUT_DIR = BASE_PATH / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GPQA_OUTPUT_PATH = OUTPUT_DIR / 'gpqa_agot_react_results.jsonl'
GPQA_TRACES_PATH = OUTPUT_DIR / 'gpqa_agot_react_detailed_traces.jsonl'
GPQA_METRICS_PATH = OUTPUT_DIR / 'gpqa_agot_metrics.json'
GPQA_CUMULATIVE_PATH = OUTPUT_DIR / 'gpqa_agot_cumulative_metrics.json'
GPQA_CHECKPOINT_PATH = OUTPUT_DIR / 'gpqa_agot_checkpoint.json'

# Model + search settings
MODEL_NAME = 'Qwen/Qwen2-7B-Instruct'
AGOT_MAX_DEPTH = 3  # Shallow depth for speed
AGOT_TOP_K = 3      # Slightly wider beam for accuracy
AGOT_NUM_ATTEMPTS = 1
REACT_MAX_STEPS = 2    # Lightweight ReAct verification
# BATCH_SIZE = 10

# Quantization toggles (GPU only)
LOAD_IN_4BIT = True   # Best speed/VRAM savings
LOAD_IN_8BIT = False  # Set True to use 8-bit instead of 4-bit

# Load model from HuggingFace
print(f"Loading {MODEL_NAME} from HuggingFace...")
print("This may take a few minutes on first run...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

    # Load with optimizations for better memory usage
    if device == "cuda":
        if LOAD_IN_4BIT:
            print("Loading model with 4-bit quantization (bnb-nf4)...")
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                trust_remote_code=True,
                load_in_4bit=True,
                device_map="auto",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                low_cpu_mem_usage=True
            )
        elif LOAD_IN_8BIT:
            print("Loading model with 8-bit quantization...")
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                trust_remote_code=True,
                load_in_8bit=True,
                device_map="auto",
                low_cpu_mem_usage=True
            )
        else:
            print("Loading model in full precision on GPU (heavier)...")
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                trust_remote_code=True,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True
            )
    else:
        print("⚠️ Running on CPU - this will be slow. Consider using Google Colab for GPU access.")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            trust_remote_code=True,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )
        model = model.to(device)

    model.eval()
    print(f"✓ Model loaded successfully on {device}")
    print(f"  4-bit: {LOAD_IN_4BIT} | 8-bit: {LOAD_IN_8BIT}")

except Exception as e:
    print(f"⚠️ Error loading model: {e}")
    print("Make sure you have enough disk space and RAM/VRAM")
    raise

print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Self-consistency: {AGOT_NUM_ATTEMPTS} attempts with majority voting")
print(f"Ready!")

Loading Qwen/Qwen2-7B-Instruct from HuggingFace...
This may take a few minutes on first run...
Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✓ Model loaded successfully on cuda
  4-bit: True | 8-bit: False
Model: Qwen/Qwen2-7B-Instruct
Device: cuda
Output dir: /kaggle/working/outputs
Self-consistency: 1 attempts with majority voting
Ready!


In [3]:
# Load GPQA Diamond
from datasets import load_dataset

print("Loading GPQA Diamond...")
gpqa_dataset = load_dataset("fingertap/GPQA-Diamond", split="test")
print(f"✓ Loaded {len(gpqa_dataset)} questions")
print(f"Fields: {gpqa_dataset.column_names}")
print(json.dumps({k: str(v)[:120] for k, v in gpqa_dataset[0].items()}, indent=2))

Loading GPQA Diamond...


test/gpqa_diamond.parquet:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/198 [00:00<?, ? examples/s]

✓ Loaded 198 questions
Fields: ['question', 'answer']
{
  "question": "Among the following exoplanets, which one has the highest density?\n\na) An Earth-mass and Earth-radius planet.\nb) A plane",
  "answer": "D"
}


## External Tools for ReAct

In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus

class ExternalToolExecutor:
    def __init__(self):
        self.search_history = []

    def search_wikipedia(self, entity: str) -> str:
        try:
            api_url = "https://en.wikipedia.org/w/api.php"
            params = {
                'action': 'query', 'format': 'json', 'titles': entity,
                'prop': 'extracts', 'explaintext': True, 'exintro': True, 'redirects': 1
            }
            r = requests.get(api_url, params=params, timeout=10)
            data = r.json()
            pages = data.get('query', {}).get('pages', {})
            if not pages:
                return f"No Wikipedia page for '{entity}'."
            page = pages[list(pages.keys())[0]]
            if 'missing' in page:
                return f"No page for '{entity}'."
            extract = page.get('extract', '')
            if not extract:
                return f"No content for '{entity}'."
            words = extract.split()
            snippet = ' '.join(words[:200])
            return snippet + ('...' if len(words) > 200 else '')
        except Exception as e:
            return f"Wikipedia search failed: {str(e)[:80]}"

    def search_web(self, query: str) -> str:
        try:
            url = f"https://html.duckduckgo.com/html/?q={quote_plus(query)}"
            headers = {'User-Agent': 'Mozilla/5.0'}
            r = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.text, features="html.parser")
            snippets = []
            for item in soup.find_all("div", {"class": "result"})[:3]:
                sn = item.find("a", {"class": "result__snippet"})
                if sn:
                    text = sn.get_text().strip()
                    if text:
                        snippets.append(text)
            if not snippets:
                for p in soup.find_all("p", limit=3):
                    text = p.get_text().strip()
                    if len(text) > 20:
                        snippets.append(text)
            if snippets:
                combined = " ".join(snippets)
                words = combined.split()
                return ' '.join(words[:150]) + ('...' if len(words) > 150 else '')
            return f"No web results for '{query}'."
        except Exception as e:
            return f"Web search failed: {str(e)[:80]}"

    def lookup_in_text(self, keyword: str, context: str) -> str:
        if not context:
            return "No context."
        sentences = context.replace('\n', ' ').split('.')
        matches = [s.strip() for s in sentences if keyword.lower() in s.lower() and len(s.strip()) > 5]
        if matches:
            joined = '. '.join(matches[:2]) + '.'
            words = joined.split()
            return ' '.join(words[:120])
        return f"'{keyword}' not found."

external_tools = ExternalToolExecutor()
print("✓ External tools ready (Wikipedia + Web search + lookup)")

✓ External tools ready (Wikipedia + Web search + lookup)


## Simplified AGoT Reasoning Engine

**Architecture**: Node-based graph with Thought → Action → Observation loop

**Key Components**:
1. **AGOTNode**: state, confidence, status, action_result, domain
2. **AGOTGraph**: nodes dict, root, deduplication, path extraction
3. **Domain Analysis**: Auto-detect problem type for better reasoning
4. **Confidence Scoring**: Track reasoning quality, prune dead ends

In [5]:
import uuid
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Tuple, Optional, Any

# ========================================
# Simplified AGoT Node Structure
# ========================================

@dataclass
class AGOTNode:
    """Simplified AGoT node with confidence scoring."""
    id: str
    state: str  # Current reasoning state/thought
    assumptions: List[str] = field(default_factory=list)
    confidence: float = 1.0
    status: str = "open"  # "open", "dead_end", "completed"
    parents: List[str] = field(default_factory=list)
    children: List[str] = field(default_factory=list)
    depth: int = 0
    action_result: Optional[str] = None
    domain: str = ""  # Physics, Chemistry, Biology, Math

@dataclass
class AGOTGraph:
    """Graph structure for AGoT reasoning."""
    nodes: Dict[str, AGOTNode] = field(default_factory=dict)
    root_id: Optional[str] = None

    def add_node(self, node: AGOTNode, parent_id: Optional[str] = None) -> bool:
        """Add node to graph, merge if duplicate exists."""
        # Check for duplicate state
        duplicate = self.find_duplicate(node.state)
        if duplicate:
            # Merge: add parent link to existing node
            if parent_id and parent_id not in duplicate.parents:
                duplicate.parents.append(parent_id)
                if parent_id in self.nodes:
                    self.nodes[parent_id].children.append(duplicate.id)
            return False  # Didn't add new node
        else:
            # Add new node
            if parent_id:
                node.parents.append(parent_id)
                if parent_id in self.nodes:
                    self.nodes[parent_id].children.append(node.id)
                    node.depth = self.nodes[parent_id].depth + 1
            self.nodes[node.id] = node
            if self.root_id is None:
                self.root_id = node.id
            return True  # Added new node

    def find_duplicate(self, state: str) -> Optional[AGOTNode]:
        """Find node with identical state."""
        for node in self.nodes.values():
            if node.state.strip().lower() == state.strip().lower():
                return node
        return None

    def get_expandable_nodes(self) -> List[AGOTNode]:
        """Get all nodes with status='open'."""
        return [n for n in self.nodes.values() if n.status == "open"]

    def get_top_k_nodes(self, k: int = 5) -> List[AGOTNode]:
        """Get top-k nodes by confidence."""
        return sorted(self.nodes.values(), key=lambda n: n.confidence, reverse=True)[:k]

    def extract_solution_path(self, goal_node_id: str) -> List[AGOTNode]:
        """Extract path from root to goal node."""
        path = []
        current_id = goal_node_id
        while current_id:
            if current_id in self.nodes:
                node = self.nodes[current_id]
                path.insert(0, node)
                current_id = node.parents[0] if node.parents else None
            else:
                break
        return path

    def summary(self, n_nodes: int = 10) -> str:
        """Get summary of top nodes."""
        lines = []
        for node in self.get_top_k_nodes(n_nodes):
            lines.append(f"[D{node.depth}] {node.state[:80]} (conf={node.confidence:.2f}, status={node.status})")
        return "\n".join(lines)

# ========================================
# LLM Generate Function (Qwen2-7B)
# ========================================

async def llm_generate(prompt: str, temperature: float = 0.1, max_tokens: int = 128) -> str:
    """Generate from Qwen2 using HuggingFace transformers (short outputs for speed)."""
    try:
        # Format prompt for Qwen2-Instruct
        messages = [
            {"role": "system", "content": "You are a helpful AI assistant skilled in PhD-level STEM reasoning."},
            {"role": "user", "content": prompt}
        ]

        # Use chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize
        inputs = tokenizer([text], return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        return response.strip()

    except Exception as e:
        print(f"⚠️ LLM error: {e}")
        return ""

# ========================================
# Domain & Problem Type Analysis
# ========================================

async def analyze_domain_and_type(question: str) -> Dict[str, str]:
    """Analyze the domain and problem type for better reasoning."""
    prompt = f"""Analyze this PhD-level question and identify:
1. Domain: Physics, Chemistry, Biology, or Math
2. Problem Type: (e.g., derivation, calculation, mechanism, pathway, proof, optimization)
3. Key Concepts: List 2-3 main concepts

Question: {question[:500]}

Format your response as:
Domain: [domain]
Type: [type]
Concepts: [concept1, concept2, concept3]"""

    response = await llm_generate(prompt, temperature=0.1, max_tokens=150)

    # Parse response
    domain = "General"
    prob_type = "Analysis"
    concepts = []

    for line in response.split('\n'):
        if line.startswith("Domain:"):
            domain = line.split(":", 1)[1].strip()
        elif line.startswith("Type:"):
            prob_type = line.split(":", 1)[1].strip()
        elif line.startswith("Concepts:"):
            concepts_str = line.split(":", 1)[1].strip()
            concepts = [c.strip() for c in concepts_str.split(',')]

    return {
        "domain": domain,
        "problem_type": prob_type,
        "key_concepts": concepts
    }

print("✓ Simplified AGoT node structure ready (with domain analysis)")

✓ Simplified AGoT node structure ready (with domain analysis)


In [6]:

# ========================================
# AGoT Engine - Simplified ReAct-Style Flow with Verification
# ========================================

class AGoTEngine:
    """Simplified AGoT engine with step verification and self-consistency voting."""

    def __init__(self, max_depth: int = 3, top_k: int = 5, num_attempts: int = 3):
        self.max_depth = max_depth
        self.top_k = top_k
        self.num_attempts = num_attempts  # For self-consistency voting
        self.metrics = {
            "nodes_created": 0,
            "nodes_evaluated": 0,
            "nodes_pruned": 0,
            "actions_performed": 0,
            "verifications_performed": 0,
            "verifications_failed": 0
        }

    async def verify_action_result(self, thought: str, action_result: str, domain: str) -> Tuple[bool, str]:
        """Verify if action result is logically sound and consistent."""
        self.metrics["verifications_performed"] += 1

        prompt = f"""Verify if this reasoning step and its result are logically sound.

Domain: {domain}
Reasoning Step: {thought}
Result/Observation: {action_result}

Check for:
1. Logical consistency
2. Correct application of principles/formulas
3. No computational errors
4. Result makes sense in context

Respond with:
Valid: YES or NO
Reason: [brief explanation if NO]

Format:
Valid: [YES/NO]
Reason: [explanation]"""

        response = await llm_generate(prompt, temperature=0.1, max_tokens=200)

        # Parse validation
        is_valid = True
        reason = "Passed verification"

        for line in response.split('\n'):
            if line.startswith("Valid:"):
                valid_str = line.split(":", 1)[1].strip().upper()
                is_valid = "YES" in valid_str or "TRUE" in valid_str
            elif line.startswith("Reason:"):
                reason = line.split(":", 1)[1].strip()

        if not is_valid:
            self.metrics["verifications_failed"] += 1

        return is_valid, reason

    async def generate_candidate_thoughts(self, node: AGOTNode, context: str, domain_info: Dict) -> List[str]:
        """Generate candidate reasoning steps based on domain and parent context."""
        domain = domain_info.get("domain", "General")
        prob_type = domain_info.get("problem_type", "Analysis")

        # Build parent context from action results
        parent_context = ""
        if node.action_result:
            parent_context = f"\nPrevious observation: {node.action_result[:200]}"

        prompt = f"""Given the current reasoning state, generate 3-5 next logical reasoning steps.

Domain: {domain}
Problem Type: {prob_type}
Current State: {node.state}{parent_context}

Question Context: {context[:300]}

Generate specific, actionable reasoning steps (one per line):
- For Math: lemma applications, equation transforms, proof steps
- For Physics: formula applications, law applications, boundary conditions
- For Chemistry: reaction steps, intermediate predictions, mechanism steps
- For Biology: pathway interactions, regulatory steps, molecular interactions

Return ONLY the reasoning steps, one per line, no numbering:"""

        response = await llm_generate(prompt, temperature=0.5, max_tokens=300)

        # Parse candidates
        candidates = []
        for line in response.split('\n'):
            line = line.strip()
            # Remove numbering
            line = re.sub(r'^\d+[\.\)]\s*', '', line)
            line = re.sub(r'^[\-\•]\s*', '', line)
            if line and len(line) > 10:
                candidates.append(line)

        return candidates[:5]  # Max 5 candidates

    async def perform_action(self, thought: str, domain: str, context: str) -> Tuple[str, bool]:
        """Perform domain-specific action and return observation."""
        self.metrics["actions_performed"] += 1

        # Domain-specific action prompt
        if domain.lower() in ["math", "mathematics"]:
            action_type = "compute, derive, or verify"
        elif domain.lower() == "physics":
            action_type = "solve equation, apply law, or simulate"
        elif domain.lower() == "chemistry":
            action_type = "predict reaction, verify mechanism, or calculate"
        elif domain.lower() == "biology":
            action_type = "verify pathway, check regulation, or analyze interaction"
        else:
            action_type = "analyze or verify"

        prompt = f"""Execute this reasoning step and provide the result.

Action: {action_type}
Reasoning Step: {thought}
Context: {context[:200]}

Provide:
1. Result/Observation (brief, factual)
2. Success: YES or NO (whether this step leads to progress)

Format:
Result: [your observation]
Success: [YES/NO]"""

        response = await llm_generate(prompt, temperature=0.3, max_tokens=250)

        # Parse result and success
        result = response
        success = True  # Default

        for line in response.split('\n'):
            if line.startswith("Result:"):
                result = line.split(":", 1)[1].strip()
            elif line.startswith("Success:"):
                success_str = line.split(":", 1)[1].strip().upper()
                success = "YES" in success_str or "TRUE" in success_str

        return result, success

    def update_confidence(self, node: AGOTNode, success: bool, verification_passed: bool = True):
        """Update node confidence based on action success and verification."""
        if success and verification_passed:
            node.confidence *= 1.1
            node.confidence = min(node.confidence, 2.0)  # Cap at 2.0
        elif not verification_passed:
            # Failed verification is worse than just unsuccessful action
            node.confidence *= 0.3
            if node.confidence < 0.15:
                node.status = "dead_end"
        else:
            node.confidence *= 0.5
            if node.confidence < 0.2:
                node.status = "dead_end"

    def prune_low_confidence_nodes(self, graph: AGOTGraph):
        """Prune nodes with low confidence, keeping top-k."""
        all_nodes = list(graph.nodes.values())
        open_nodes = [n for n in all_nodes if n.status == "open"]

        if len(open_nodes) <= self.top_k:
            return  # No need to prune

        # Sort by confidence
        open_nodes.sort(key=lambda n: n.confidence, reverse=True)

        # Mark low-confidence nodes as dead_end
        for node in open_nodes[self.top_k:]:
            node.status = "dead_end"
            self.metrics["nodes_pruned"] += 1

    def check_goal(self, node: AGOTNode, question: str) -> bool:
        """Check if node represents a valid solution."""
        # Simple heuristic: node has high confidence and contains answer indicator
        if node.confidence < 0.7:
            return False

        state_lower = node.state.lower()
        result_lower = (node.action_result or "").lower()

        # Check for answer indicators
        answer_indicators = [
            "answer is", "solution is", "result is",
            "option a", "option b", "option c", "option d",
            "therefore a", "therefore b", "therefore c", "therefore d"
        ]

        for indicator in answer_indicators:
            if indicator in state_lower or indicator in result_lower:
                return True

        return False

    async def run_single_attempt(self, question: str, domain_info: Dict, attempt_num: int = 1) -> Tuple[str, AGOTGraph, Dict]:
        """Execute single AGoT reasoning attempt."""
        # Initialize graph with root node
        root = AGOTNode(
            id=str(uuid.uuid4()),
            state=f"Problem: {question[:200]}",
            domain=domain_info["domain"],
            depth=0
        )
        graph = AGOTGraph()
        graph.add_node(root)
        attempt_metrics = {"nodes_created": 1, "nodes_evaluated": 0, "actions_performed": 0}

        # Main AGoT loop
        for depth in range(self.max_depth):
            # Select expandable nodes
            expandable = graph.get_expandable_nodes()
            if not expandable:
                break

            # Limit to top-k nodes to prevent explosion
            expandable = sorted(expandable, key=lambda n: n.confidence, reverse=True)[:self.top_k]

            # For each expandable node
            for node in expandable:
                # Generate candidate thoughts
                candidates = await self.generate_candidate_thoughts(node, question, domain_info)

                # Create child nodes
                for candidate in candidates:
                    new_node = AGOTNode(
                        id=str(uuid.uuid4()),
                        state=candidate,
                        domain=domain_info["domain"],
                        assumptions=node.assumptions.copy()
                    )

                    # Add to graph (will merge if duplicate)
                    added = graph.add_node(new_node, node.id)
                    if added:
                        attempt_metrics["nodes_created"] += 1

            # Perform actions and verify for new nodes
            new_nodes = [n for n in graph.nodes.values() if n.status == "open" and n.action_result is None]
            for node in new_nodes:
                if node.id == root.id:
                    continue  # Skip root

                # Perform action
                result, success = await self.perform_action(
                    node.state,
                    domain_info["domain"],
                    question
                )

                node.action_result = result
                attempt_metrics["nodes_evaluated"] += 1
                attempt_metrics["actions_performed"] += 1

                # Verify action result
                verification_passed = True
                if success:  # Only verify if action claims success
                    verification_passed, verify_reason = await self.verify_action_result(
                        node.state,
                        result,
                        domain_info["domain"]
                    )
                    if not verification_passed:
                        node.action_result += f" [VERIFICATION FAILED: {verify_reason}]"

                # Update confidence
                self.update_confidence(node, success, verification_passed)

            # Prune low-confidence nodes
            self.prune_low_confidence_nodes(graph)

            # Check if any node reached goal
            goal_nodes = [n for n in graph.nodes.values() if self.check_goal(n, question)]
            if goal_nodes:
                # Sort by confidence and mark best as completed
                goal_nodes.sort(key=lambda n: n.confidence, reverse=True)
                goal_nodes[0].status = "completed"
                break

        # Extract solution
        completed_nodes = [n for n in graph.nodes.values() if n.status == "completed"]
        if completed_nodes:
            best_node = max(completed_nodes, key=lambda n: n.confidence)
            solution_path = graph.extract_solution_path(best_node.id)
            final_answer = await self.synthesize_solution(solution_path, question)
        else:
            # Fallback: use highest confidence node
            top_nodes = graph.get_top_k_nodes(1)
            if top_nodes:
                solution_path = graph.extract_solution_path(top_nodes[0].id)
                final_answer = await self.synthesize_solution(solution_path, question)
            else:
                final_answer = "Unable to determine answer"

        return final_answer, graph, attempt_metrics

    async def run(self, question: str) -> Tuple[str, AGOTGraph, Dict]:
        """Execute AGoT reasoning with self-consistency voting."""
        # Step 1: Analyze domain and problem type
        domain_info = await analyze_domain_and_type(question)

        # Step 2: Run multiple independent attempts
        attempts = []
        all_graphs = []

        print(f"  Running {self.num_attempts} independent reasoning attempts...")
        for i in range(self.num_attempts):
            try:
                final_answer, graph, attempt_metrics = await self.run_single_attempt(
                    question, domain_info, attempt_num=i+1
                )
                attempts.append({
                    "answer": final_answer,
                    "graph": graph,
                    "metrics": attempt_metrics
                })
                all_graphs.append(graph)

                # Update global metrics
                for key in attempt_metrics:
                    if key in self.metrics:
                        self.metrics[key] += attempt_metrics[key]

            except Exception as e:
                print(f"  ⚠️ Attempt {i+1} failed: {str(e)[:60]}")
                continue

        # Step 3: Vote on final answer (self-consistency)
        if not attempts:
            return "Unable to determine answer", AGOTGraph(), self.metrics

        # Extract all answers and vote
        answers = [extract_choice_letter(a["answer"], "?") for a in attempts]
        answer_counts = {}
        for ans in answers:
            answer_counts[ans] = answer_counts.get(ans, 0) + 1

        # Get majority vote
        voted_answer = max(answer_counts.items(), key=lambda x: x[1])[0]
        vote_confidence = answer_counts[voted_answer] / len(answers)

        # Find best graph that produced the voted answer
        best_graph = None
        best_confidence = 0
        for attempt in attempts:
            attempt_answer = extract_choice_letter(attempt["answer"], "?")
            if attempt_answer == voted_answer:
                # Get top node confidence from this graph
                top_nodes = attempt["graph"].get_top_k_nodes(1)
                if top_nodes and top_nodes[0].confidence > best_confidence:
                    best_confidence = top_nodes[0].confidence
                    best_graph = attempt["graph"]

        # Use best graph or first graph as fallback
        final_graph = best_graph if best_graph else all_graphs[0]

        # Create final answer with voting info
        final_answer = f"The answer is {voted_answer} (voted {answer_counts[voted_answer]}/{len(answers)} attempts, confidence={vote_confidence:.1%})"

        self.metrics["voting_confidence"] = vote_confidence
        self.metrics["vote_distribution"] = answer_counts

        return final_answer, final_graph, self.metrics

    async def synthesize_solution(self, path: List[AGOTNode], question: str) -> str:
        """Synthesize final solution from reasoning path."""
        path_summary = "\n".join([
            f"Step {i+1}: {node.state}\n  → {node.action_result or 'N/A'}"
            for i, node in enumerate(path) if node.action_result
        ])

        prompt = f"""Based on this reasoning path, provide the final answer.

Question: {question}

Reasoning Path:
{path_summary[:800]}

IMPORTANT: End your response with exactly one of these:
- "The answer is A"
- "The answer is B"
- "The answer is C"
- "The answer is D"

Provide brief justification then state the answer:"""

        response = await llm_generate(prompt, temperature=0.2, max_tokens=300)
        return response.strip()

print("✓ AGoT engine ready with step verification + self-consistency voting")
print("  Features: Verify each action | Vote across 3 independent attempts")

✓ AGoT engine ready with step verification + self-consistency voting
  Features: Verify each action | Vote across 3 independent attempts


## ReAct Verification

In [7]:
def parse_action(text: str) -> tuple:
    patterns = [
        (r"search\[(.+?)\]", "search"),
        (r"lookup\[(.+?)\]", "lookup"),
        (r"finish\[([A-D])\]", "finish"),
    ]
    t = text.lower()
    for pattern, action_type in patterns:
        m = re.search(pattern, t, re.IGNORECASE | re.DOTALL)
        if m:
            return action_type, m.group(1).strip()
    return None, None


def normalize_choice_letter(text: str, fallback: str = "?") -> str:
    """Extract a top-level A/B/C/D letter from free text; fallback if none."""
    if not text:
        return fallback
    patterns = [
        r"answer\s+is\s+([A-D])",
        r"option\s+([A-D])",
        r"([A-D])\)",
        r"\b([A-D])\b",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            letter = m.group(1).upper()
            if letter in ["A", "B", "C", "D"]:
                return letter
    return fallback


async def react_verify_answer(question: str, agot_answer: str, agot_graph: AGOTGraph, max_steps: int = REACT_MAX_STEPS) -> dict:
    """ReAct loop with AGoT-style thinking driving each step."""
    steps = []
    current_answer = normalize_choice_letter(agot_answer, agot_answer)

    def format_observation_history():
        if not steps:
            return "None"
        return "\n".join(
            [f"Observation {s['iteration']}: {s['observation']}" for s in steps if s.get("observation")]
        )

    try:
        for i in range(1, max_steps + 1):
            history_block = format_observation_history()
            prompt = (
                "You are performing ReAct verification using AGoT-style thinking. "
                "Follow the canonical Thought->Action->Observation pattern.\n\n"
                f"Question:\n{question}\n\n"
                f"Current hypothesis: {current_answer}\n"
                f"AGoT summary:\n{agot_graph.summary(150)}\n\n"
                f"Observations so far:\n{history_block}\n\n"
                f"Thinking {i}: <reason briefly>\n"
                f"Action {i}: <search[term] | lookup[keyword] | finish[A/B/C/D]>"
            )

            thought_action = await llm_generate(prompt, temperature=0.2, max_tokens=220)

            # Extract thinking and action lines in ReAct format
            thinking = thought_action
            action_text = thought_action
            if re.search(rf"action\s+{i}\s*:", thought_action, flags=re.IGNORECASE):
                parts = re.split(rf"action\s+{i}\s*:", thought_action, flags=re.IGNORECASE)
                thinking = parts[0].strip()
                action_text = parts[1].strip() if len(parts) > 1 else ""
            else:
                # Try to split on a newline if user returned two lines
                lines = [l.strip() for l in thought_action.splitlines() if l.strip()]
                if len(lines) >= 2:
                    thinking, action_text = lines[0], lines[1]

            action_type, parameter = parse_action(action_text)

            if action_type == "finish":
                final = normalize_choice_letter(parameter, current_answer)
                observation = f"Finish with answer {final}."
                current_answer = final
            elif action_type == "search":
                observation = external_tools.search_wikipedia(parameter)
            elif action_type == "lookup":
                observation = external_tools.lookup_in_text(parameter, steps[-1]["observation"] if steps else "")
            else:
                observation = "No valid action. Use search[], lookup[], or finish[]."

            steps.append({
                "iteration": i,
                "thinking": thinking,
                "action": f"{action_type}[{parameter}]" if action_type else action_text,
                "observation": observation,
            })

            if action_type == "finish":
                break
    except Exception as e:
        steps.append({"iteration": len(steps) + 1, "thinking": "error", "action": "error", "observation": str(e)[:120]})

    current_answer = normalize_choice_letter(current_answer, agot_answer)
    if current_answer not in ["A", "B", "C", "D"]:
        current_answer = normalize_choice_letter(agot_answer, agot_answer)

    return {"steps": steps, "final_answer": current_answer}

print("✓ ReAct verification ready (AGoT-driven thinking)")

✓ ReAct verification ready (AGoT-driven thinking)


## AGoT + ReAct Solver

In [8]:
import asyncio

# Helper: robustly extract A/B/C/D from text
def extract_choice_letter(text: str, fallback: str = "?") -> str:
    if not text:
        return fallback

    text_upper = text.upper()

    # Priority-ordered patterns for extracting choice
    patterns = [
        r"ANSWER\s+IS\s+([A-D])",
        r"ANSWER\s*[:=]\s*([A-D])",
        r"FINAL\s+ANSWER\s*[:=]?\s*([A-D])",
        r"CHOICE\s+([A-D])",
        r"OPTION\s+([A-D])",
        r"THEREFORE\s+([A-D])",
        r"([A-D])\s*[\.\):\-]",
        r"\b([A-D])\b(?=[\s\.\,\!\?]|$)",
    ]

    for pat in patterns:
        m = re.search(pat, text_upper)
        if m:
            letter = m.group(1).upper()
            if letter in ["A", "B", "C", "D"]:
                return letter

    # Last resort: find any standalone letter A-D
    last_match = re.findall(r"[A-D]", text_upper)
    if last_match:
        for letter in reversed(last_match):
            if letter in ["A", "B", "C", "D"]:
                return letter

    return fallback


async def agot_react_solve_question(example: dict, agot_engine: AGoTEngine) -> dict:
    """Solve question using simplified AGoT (with voting) + ReAct verification."""
    question = example.get('question', '')
    correct_answer = example.get('correct_answer', '')
    index = example.get('index', -1)

    try:
        # PHASE 1: AGoT Reasoning with Self-Consistency Voting
        print(f"\n[Q{index}] Starting AGoT reasoning with self-consistency voting...")
        try:
            agot_final, agot_graph, agot_metrics = await agot_engine.run(question)
            print(f"[Q{index}] AGoT: {agot_metrics.get('nodes_created', 0)} nodes, " +
                  f"{agot_metrics.get('verifications_performed', 0)} verifications, " +
                  f"{agot_metrics.get('verifications_failed', 0)} failed")
            print(f"[Q{index}] Voting: {agot_metrics.get('vote_distribution', {})} " +
                  f"(confidence: {agot_metrics.get('voting_confidence', 0):.1%})")
        except Exception as e:
            print(f"⚠️ AGoT failed on Q{index}: {str(e)[:80]}")
            agot_final = f"AGoT failed: {str(e)[:100]}"
            agot_graph = AGOTGraph()
            agot_metrics = {'nodes_created': 0, 'nodes_evaluated': 0}

        # Extract A/B/C/D from AGoT synthesis
        agot_answer = extract_choice_letter(agot_final, "?")

        # Build AGoT step summary
        agot_steps = [{
            "phase": "agot_reasoning_with_voting",
            "domain": agot_graph.nodes[agot_graph.root_id].domain if agot_graph.root_id else "Unknown",
            "num_attempts": agot_engine.num_attempts,
            "voting_confidence": agot_metrics.get('voting_confidence', 0),
            "vote_distribution": agot_metrics.get('vote_distribution', {}),
            "nodes_created": agot_metrics.get('nodes_created', 0),
            "nodes_evaluated": agot_metrics.get('nodes_evaluated', 0),
            "actions_performed": agot_metrics.get('actions_performed', 0),
            "verifications_performed": agot_metrics.get('verifications_performed', 0),
            "verifications_failed": agot_metrics.get('verifications_failed', 0),
            "nodes_pruned": agot_metrics.get('nodes_pruned', 0),
            "final_synthesis": agot_final[:300],
            "extracted_answer": agot_answer,
            "graph_summary": agot_graph.summary(8)
        }]

        # PHASE 2: ReAct Verification (external tools for fact-checking)
        print(f"[Q{index}] Starting ReAct verification...")
        try:
            react_result = await react_verify_answer(question, agot_answer, agot_graph, max_steps=REACT_MAX_STEPS)
            react_steps = react_result['steps']
            final_answer = extract_choice_letter(react_result['final_answer'], agot_answer)
            print(f"[Q{index}] ReAct completed {len(react_steps)} steps, final: {final_answer}")
        except Exception as e:
            print(f"⚠️ ReAct failed on Q{index}: {str(e)[:80]}")
            react_steps = [{"iteration": 0, "thinking": "error", "action": "error", "observation": str(e)[:100]}]
            final_answer = agot_answer  # Fall back to AGoT answer

        # Build comprehensive trace
        trace_lines = [
            "=== PHASE 1: AGoT REASONING with SELF-CONSISTENCY VOTING ===",
            f"Domain: {agot_steps[0].get('domain', 'Unknown')}",
            f"Attempts: {agot_engine.num_attempts} independent reasoning paths",
            f"Vote Distribution: {agot_metrics.get('vote_distribution', {})}",
            f"Voting Confidence: {agot_metrics.get('voting_confidence', 0):.1%}",
            "",
            f"Total Nodes: {agot_metrics.get('nodes_created', 0)} created, {agot_metrics.get('nodes_evaluated', 0)} evaluated",
            f"Actions: {agot_metrics.get('actions_performed', 0)} performed",
            f"Verifications: {agot_metrics.get('verifications_performed', 0)} performed, {agot_metrics.get('verifications_failed', 0)} failed ✓",
            f"Pruned: {agot_metrics.get('nodes_pruned', 0)} low-confidence nodes",
            "",
            "Top Reasoning Paths (from best voted attempt):",
            agot_graph.summary(8),
            "",
            f"AGoT Final Synthesis:\n{agot_final[:400]}...",
            f"AGoT Extracted Answer: {agot_answer}",
            "",
            "=== PHASE 2: ReAct VERIFICATION (External Fact-Checking) ==="
        ]

        for s in react_steps:
            trace_lines.append(f"Step {s.get('iteration', 0)}:")
            trace_lines.append(f"  Thinking: {s.get('thinking', 'N/A')[:100]}")
            trace_lines.append(f"  Action: {s.get('action', 'N/A')}")
            trace_lines.append(f"  Observation: {s.get('observation', 'N/A')[:150]}")

        trace_lines.append(f"\n=== FINAL ANSWER: {final_answer} ===")
        trace_lines.append(f"Correct Answer: {correct_answer}")
        trace_lines.append(f"Result: {'✓ CORRECT' if final_answer == correct_answer else '✗ INCORRECT'}")

        trace = "\n".join(trace_lines)

        return {
            "index": index,
            "question": question,
            "correct_answer": correct_answer,
            "react_answer": final_answer if final_answer != "?" else agot_answer,
            "is_correct": (final_answer if final_answer != "?" else agot_answer) == correct_answer,
            "react_trace": trace,
            "steps": {
                "agot_steps": agot_steps,
                "react_steps": react_steps,
                "agot_metrics": agot_metrics
            },
            "agot_answer": agot_answer,
            "voting_info": {
                "distribution": agot_metrics.get('vote_distribution', {}),
                "confidence": agot_metrics.get('voting_confidence', 0)
            }
        }

    except Exception as e:
        print(f"⚠️ Critical error solving Q{index}: {str(e)[:100]}")
        return {
            "index": index,
            "question": question,
            "correct_answer": correct_answer,
            "react_answer": "?",
            "is_correct": False,
            "react_trace": f"CRITICAL ERROR: {str(e)[:300]}",
            "steps": {"agot_steps": [], "react_steps": [], "agot_metrics": {}},
            "agot_answer": "?",
        }

# Initialize AGoT engine with self-consistency voting
agot_engine = AGoTEngine(max_depth=AGOT_MAX_DEPTH, top_k=AGOT_TOP_K, num_attempts=AGOT_NUM_ATTEMPTS)
print(f"✓ AGoT+ReAct solver ready (max_depth={AGOT_MAX_DEPTH}, top_k={AGOT_TOP_K}, voting={AGOT_NUM_ATTEMPTS} attempts)")
print(f"✓ Using Qwen2-7B-Instruct with:")
print(f"  - Domain-aware reasoning")
print(f"  - Step-by-step verification ✓")
print(f"  - Self-consistency voting (3 attempts) ✓")

✓ AGoT+ReAct solver ready (max_depth=3, top_k=3, voting=1 attempts)
✓ Using Qwen2-7B-Instruct with:
  - Domain-aware reasoning
  - Step-by-step verification ✓
  - Self-consistency voting (3 attempts) ✓


## Batch Evaluation with Checkpoints

In [ ]:
# Prepare data
formatted_data = []
for idx, ex in enumerate(gpqa_dataset):
    q = ex.get('question', '')
    ans = (ex.get('answer','') or '').strip().upper()
    if len(ans) > 1:
        m = re.search(r'([A-D])', ans)
        if m:
            ans = m.group(1)
    formatted_data.append({
        'index': idx,
        'question': q,
        'correct_answer': ans
    })
print(f"Prepared {len(formatted_data)} examples")

# Load checkpoint
checkpoint_data = {"evaluated_indices": set(), "accumulated_results": []}
if GPQA_CHECKPOINT_PATH.exists():
    try:
        with open(GPQA_CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
            saved = json.load(f)
            checkpoint_data['evaluated_indices'] = set(saved.get('evaluated_indices', []))
            checkpoint_data['accumulated_results'] = saved.get('accumulated_results', [])
        print(f"✓ Checkpoint: {len(checkpoint_data['evaluated_indices'])} already evaluated")
    except:
        print("⚠️ Checkpoint corrupted, starting fresh")

# Compute remaining
all_indices = set(range(len(formatted_data)))
remaining = sorted(all_indices - checkpoint_data['evaluated_indices'])

# Resume from index 50 to last
batch_indices = [idx for idx in remaining if idx >= 156]
print(f"\n🔄 Running from index 0 to last: {len(batch_indices)} questions")
if batch_indices:
    print(f"Starting at index {batch_indices[0]}, ending at {batch_indices[-1]}")
print(f"Progress: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)} already done")

if not batch_indices:
    print("✓ All examples from index 50 onwards already evaluated!")
    results = checkpoint_data['accumulated_results']
else:

    # Run async batch
    async def run_batch():
        results = []
        for idx in tqdm(batch_indices, desc="AGoT+ReAct"):
            try:
                result = await agot_react_solve_question(formatted_data[idx], agot_engine)
                results.append(result)
                checkpoint_data['evaluated_indices'].add(idx)
                checkpoint_data['accumulated_results'].append(result)
                print("pls stay active")
                # Save incrementally
                with open(GPQA_OUTPUT_PATH, 'a', encoding='utf-8') as f:
                    json.dump({
                        "index": result['index'],
                        "question": result['question'],
                        "answer": result['react_answer'],
                        "correct_answer": result['correct_answer'],
                        "is_correct": result['is_correct'],
                        "react_trace": result['react_trace'],
                        "timestamp": datetime.now().isoformat()
                    }, f, ensure_ascii=False)
                    f.write("\n")
                    print("pls stay active")

                with open(GPQA_TRACES_PATH, 'a', encoding='utf-8') as f:
                    json.dump(result, f, ensure_ascii=False)
                    f.write("\n")
                    print("Printing to keep the session active in kaggle...")
                print("pls stay active")
                # Save checkpoint after EACH question (not just at end of batch)
                with open(GPQA_CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
                    json.dump({
                        'evaluated_indices': sorted(list(checkpoint_data['evaluated_indices'])),
                        'accumulated_results': checkpoint_data['accumulated_results'][-50:],  # keep last 50
                        'timestamp': datetime.now().isoformat()
                    }, f, ensure_ascii=False, indent=2)
                    print("pls stay active")
                print("Printing to keep the session active in kaggle...")
            except Exception as e:
                print(f"⚠️ Error on index {idx}: {str(e)[:100]}")
                continue

        return results

    # Execute - wrap in asyncio.run() for Jupyter/Kaggle compatibility
    import asyncio
    try:
        # Try IPython's native async support first
        results = await run_batch()
    except:
        # Fallback to asyncio.run() if await doesn't work at top level
        results = asyncio.run(run_batch())

    if results:
        correct_count = sum(1 for r in results if r['is_correct'])
        batch_accuracy = correct_count / len(results) * 100
        print(f"\n✓ Batch complete: {correct_count}/{len(results)} correct ({batch_accuracy:.1f}%)")
    else:
        results = []
        print("⚠️ No results generated")

print(f"Total: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)}")

Prepared 198 examples

🔄 Running from index 0 to last: 42 questions
Starting at index 156, ending at 197
Progress: 0/198 already done


AGoT+ReAct:   0%|          | 0/42 [00:00<?, ?it/s]


[Q156] Starting AGoT reasoning with self-consistency voting...
  Running 1 independent reasoning attempts...
[Q156] AGoT: 36 nodes, 32 verifications, 2 failed
[Q156] Voting: {'D': 1} (confidence: 100.0%)
[Q156] Starting ReAct verification...


AGoT+ReAct:   2%|▏         | 1/42 [15:07<10:20:13, 907.65s/it]

[Q156] ReAct completed 1 steps, final: D
pls stay active
pls stay active
Printing to keep the session active in kaggle...
pls stay active
pls stay active
Printing to keep the session active in kaggle...

[Q157] Starting AGoT reasoning with self-consistency voting...
  Running 1 independent reasoning attempts...
[Q157] AGoT: 57 nodes, 52 verifications, 3 failed
[Q157] Voting: {'A': 1} (confidence: 100.0%)
[Q157] Starting ReAct verification...


...

[Q170] ReAct completed 1 steps, final: A
pls stay active
pls stay active
Printing to keep the session active in kaggle...
pls stay active
pls stay active
Printing to keep the session active in kaggle...

[Q171] Starting AGoT reasoning with self-consistency voting...
  Running 1 independent reasoning attempts...
[Q171] AGoT: 508 nodes, 472 verifications, 49 failed
[Q171] Voting: {'C': 1} (confidence: 100.0%)
[Q171] Starting ReAct verification...


AGoT+ReAct:  38%|███▊      | 16/42 [3:33:33<5:45:58, 798.39s/it]

[Q171] ReAct completed 2 steps, final: C
pls stay active
pls stay active
Printing to keep the session active in kaggle...
pls stay active
pls stay active
Printing to keep the session active in kaggle...

[Q172] Starting AGoT reasoning with self-consistency voting...
  Running 1 independent reasoning attempts...
[Q172] AGoT: 542 nodes, 504 verifications, 54 failed
[Q172] Voting: {'A': 1} (confidence: 100.0%)
[Q172] Starting ReAct verification...


AGoT+ReAct:  40%|████      | 17/42 [3:50:03<5:56:40, 856.01s/it]

[Q172] ReAct completed 2 steps, final: A
pls stay active
pls stay active
Printing to keep the session active in kaggle...
pls stay active
pls stay active
Printing to keep the session active in kaggle...

[Q173] Starting AGoT reasoning with self-consistency voting...
  Running 1 independent reasoning attempts...


## Metrics & Analysis

In [ ]:
# Calculate metrics
results_df = pd.DataFrame(results)
correct_results = results_df[results_df['is_correct'] == True]
incorrect_results = results_df[results_df['is_correct'] == False]

batch_correct = len(correct_results)
total_batch = len(results_df)
batch_accuracy = batch_correct / total_batch * 100 if total_batch else 0

print("\n" + "="*60)
print("BATCH ANALYSIS")
print("="*60)
print(f"Correct: {batch_correct}/{total_batch} ({batch_accuracy:.1f}%)")

# Diagnostic: Count "?" answers
unknown_answers = results_df[results_df['react_answer'] == '?']
unknown_agot = results_df[results_df['agot_answer'] == '?']
print(f"\nDiagnostics:")
print(f"  Questions with ReAct '?' answers: {len(unknown_answers)}/{total_batch} ({len(unknown_answers)/total_batch*100:.1f}%)")
print(f"  Questions with AGoT '?' answers: {len(unknown_agot)}/{total_batch} ({len(unknown_agot)/total_batch*100:.1f}%)")

if len(unknown_answers) > 0:
    print(f"\nSample questions with '?' ReAct answers (first 3):")
    for _, row in unknown_answers.head(3).iterrows():
        print(f"  Q: {row['question'][:80]}...")
        print(f"    AGoT: {row['agot_answer']} | ReAct: {row['react_answer']} | Gold: {row['correct_answer']}")
        print(f"    Trace: {row['react_trace'][:150]}...")

if len(incorrect_results) > 0:
    print("\nSample incorrect (first 3):")
    for _, row in incorrect_results.head(3).iterrows():
        print(f"  Q: {row['question'][:90]}...")
        print(f"  Model: {row['react_answer']} | Gold: {row['correct_answer']}")

# Cumulative metrics
all_eval = len(checkpoint_data['evaluated_indices'])
cumulative_stats = {'total_all_batches': all_eval, 'correct_all_batches': 0, 'batches_completed': 0}
if GPQA_CUMULATIVE_PATH.exists():
    try:
        with open(GPQA_CUMULATIVE_PATH, 'r') as f:

---
## 📝 Next Steps

**To continue processing:**
1. Re-run cell 15 (Batch Evaluation) to process next 10 questions
2. Checkpoints are saved automatically - you won't lose progress
3. Each run saves:
   - `gpqa_agot_react_results.jsonl` - Results summary
   - `gpqa_agot_react_detailed_traces.jsonl` - Full reasoning traces
   - `gpqa_agot_checkpoint.json` - Progress checkpoint
   - `gpqa_agot_metrics.json` - Accuracy metrics

**Download outputs:**
- Click folder icon (left sidebar) → `outputs/` → Download files before session ends

**Estimated time:** ~20-30 batches to complete all 198 questions (3-6 hours on GPU)